# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is described via a Croissant schema at the following URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` and pandas are installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(metadata.name + ':')
print(metadata.description)

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` references.

Let's enumerate all Record Sets in the dataset and their fields. All entities are referenced by their `@id`.

In [ ]:
# List all record sets with their @id and fields

record_sets = list(dataset.record_sets())

if not record_sets:
    print('No record sets found in the schema. Attempting to infer from files (distributions) attached to the dataset.')
    # If no record sets are defined, list the available distributions (files) with @id
    if hasattr(metadata, 'distribution'):
        print('Available data distributions:')
        for dist in metadata.distribution:
            print(f"  @id: {dist['@id']}")
    else:
        print('No distributions are defined in the metadata.')
else:
    for rs in record_sets:
        print(f"RecordSet Name: {rs.name}")
        print(f"  @id: {rs.id}")
        print(f"  Fields:")
        for f in rs.fields:
            print(f"    - {f.id}")
        print()

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis.
Entities are referenced by their `@id` per the schema.

If the dataset doesn't define record sets (as is the case here), we will attempt to infer DataFrames from its distributions (i.e., CSV files or similar tabular resources).

In [ ]:
# Extract available tabular data by RecordSet @id or by distribution @id as fallback

dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets()]

# If no record sets, fall back to using distribution @id for tabular data
if not record_set_ids:
    # For demonstration, collect all distributions
    from mlcroissant._dataset import Distribution
    distribs = metadata.distribution if hasattr(metadata, 'distribution') else []
    for d in distribs:
        dist_id = d['@id']
        try:
            print(f"Attempting to read data from distribution @id: {dist_id}")
            records = list(dataset.records(distribution=dist_id))
            if records:
                dataframes[dist_id] = pd.DataFrame(records)
                print(f"Loaded {len(dataframes[dist_id])} records from {dist_id}.")
            else:
                print(f"No records found for distribution {dist_id}.")
        except Exception as e:
            print(f"Skipping {dist_id}: {e}")

    if dataframes:
        # Pick the first DataFrame for preview
        main_id = list(dataframes.keys())[0]
        print(f"\nColumns in DataFrame from {main_id}:\n{dataframes[main_id].columns.tolist()}")
        display(dataframes[main_id].head())
else:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}.")
        print(f"Fields: {df.columns.tolist()}")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
All columns referenced are by their `@id` as available in the data.

In [ ]:
# Identify the DataFrame and select numeric fields for EDA

if dataframes:
    # Pick the primary DataFrame to explore
    main_id = list(dataframes.keys())[0]
    df = dataframes[main_id].copy()
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field: {numeric_field_id}")
        # Filter records with the numeric field greater than a sample threshold (e.g., 0 for coefficients, 10 for counts)
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold}")
        display(filtered_df.head())
        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized column '{norm_col}' added:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
        # Attempt groupby using a categorical field if present
        cat_cols = df.select_dtypes(include=['object']).columns.tolist()
        group_field_id = None
        for col in cat_cols:
            if col != numeric_field_id and filtered_df[col].nunique() < len(filtered_df) // 2:
                group_field_id = col
                break
        if group_field_id:
            print(f"Grouping by field '@id': {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            display(grouped_df.head())
        else:
            print("No suitable categorical group field found for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No DataFrame loaded for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields using referenced `@id`s.

In [ ]:
# Example visualization: Histogram of a numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    main_id = list(dataframes.keys())[0]
    df = dataframes[main_id]
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if numeric_cols:
        field_id = numeric_cols[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[field_id].dropna(), bins=30, kde=True)
        plt.title(f'Histogram of {field_id}')
        plt.xlabel(field_id)
        plt.ylabel('Frequency')
        plt.show()
    else:
        print("No numeric field to visualize.")
else:
    print("No DataFrame available for visualization.")

## 6. Conclusion
In this notebook, we loaded, explored, and performed basic analysis and visualization on the FAIR^2 dataset's tabular outputs using only entity `@id`s throughout. For further work, you can:
- Explore dataset columns and field `@id`s in more detail
- Apply more advanced processing or modeling
- Chain with downstream ML workflows referencing metadata via Croissant

Refer to the dataset's Croissant JSON-LD for complete field documentation.